# 🔧 NB2 — Feature Engineering Avancé
## ImmoPredict SN
GPS · Distances POI · NLP Features · Target Encoding Bayésien

In [9]:
import pandas as pd, numpy as np, re, json, math, warnings, os
warnings.filterwarnings('ignore')
df = pd.read_csv('../data/dataset_clean.csv')
print(f'✅ {len(df):,} annonces, {df.shape[1]} colonnes')

✅ 9,376 annonces, 18 colonnes


In [10]:
# ══ 1. GPS PAR QUARTIER ══════════════════════════════
CITY_GPS = {
    'almadies':(14.745,-17.510),'ngor':(14.749,-17.514),'yoff':(14.758,-17.490),
    'ouakam':(14.724,-17.494),'mermoz':(14.710,-17.475),'fann':(14.696,-17.460),
    'plateau':(14.693,-17.447),'sacre coeur':(14.720,-17.461),
    'sacré-coeur':(14.720,-17.461),'sacre-coeur':(14.720,-17.461),
    'vdn':(14.730,-17.470),'point e':(14.694,-17.460),
    'sicap':(14.712,-17.462),'liberte':(14.715,-17.463),
    'liberté':(14.715,-17.463),'hlm':(14.713,-17.459),
    'medina':(14.695,-17.456),'grand yoff':(14.736,-17.467),
    'dieuppeul':(14.714,-17.457),'patte d oie':(14.725,-17.460),
    'nord foire':(14.742,-17.465),'parcelles':(14.748,-17.451),
    'hann':(14.726,-17.428),'colobane':(14.699,-17.454),
    'biscuiterie':(14.710,-17.455),'gueule tapee':(14.695,-17.453),
    'pikine':(14.755,-17.395),'guediawaye':(14.778,-17.393),
    'thiaroye':(14.755,-17.370),'yeumbeul':(14.765,-17.348),
    'keur massar':(14.765,-17.340),'mbao':(14.740,-17.320),
    'rufisque':(14.716,-17.274),'bargny':(14.696,-17.239),
    'dakar':(14.693,-17.447),'thies':(14.791,-16.926),
    'thiès':(14.791,-16.926),'mbour':(14.368,-16.965),
    'saly':(14.454,-17.012),'diamniadio':(14.727,-17.184),
    'kaolack':(14.150,-16.083),'saint-louis':(16.018,-16.490),
    'ziguinchor':(12.583,-16.267),'touba':(14.847,-15.883),
    'default':(14.693,-17.447),
}

def get_gps(city):
    if pd.isna(city): return CITY_GPS['default']
    k = str(city).lower().strip()
    if k in CITY_GPS: return CITY_GPS[k]
    for key,c in sorted(CITY_GPS.items(), key=lambda x: len(x[0]), reverse=True):
        if key in k: return c
    return CITY_GPS['default']

df['gps'] = df['city_clean'].apply(get_gps)
df['lat'] = df['gps'].apply(lambda x: x[0])
df['lon'] = df['gps'].apply(lambda x: x[1])
# GPS réels (coinafrique + dakarvente)
mask = df['latitude'].notna() & df['latitude'].between(12,17)
df.loc[mask,'lat'] = df.loc[mask,'latitude']
df.loc[mask,'lon'] = df.loc[mask,'longitude']
df.drop('gps',axis=1,inplace=True)
print(f'GPS: {mask.sum()} réels + {(~mask).sum()} par quartier')

GPS: 4319 réels + 5057 par quartier


In [11]:
# ══ 2. DISTANCES POI ═════════════════════════════════
POI = {
    'dist_mer':      (14.693,-17.459),
    'dist_centre':   (14.693,-17.447),
    'dist_aeroport': (14.741,-17.490),
    'dist_aibd':     (14.738,-17.091),
    'dist_port':     (14.672,-17.427),
    'dist_ucad':     (14.692,-17.464),
    'dist_vdn':      (14.730,-17.470),
}

def hav(lat1,lon1,lat2,lon2):
    R=6371.0
    p1,p2=math.radians(lat1),math.radians(lat2)
    dp,dl=math.radians(lat2-lat1),math.radians(lon2-lon1)
    a=math.sin(dp/2)**2+math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return R*2*math.atan2(math.sqrt(a),math.sqrt(1-a))

for col,(la,lo) in POI.items():
    df[col] = df.apply(lambda r: hav(r['lat'],r['lon'],la,lo), axis=1)
    df[f'log_{col}'] = np.log1p(df[col])

PREMIUM = {'almadies','ngor','mermoz','fann','plateau','sacre coeur','sacré-coeur','point e','vdn','yoff'}
PERIPH  = {'pikine','guediawaye','thiaroye','yeumbeul','keur massar','mbao','rufisque','bargny'}
def get_zone(c):
    c=str(c or '').lower()
    if any(z in c for z in PREMIUM): return 2
    if any(z in c for z in PERIPH):  return 0
    return 1
df['zone']       = df['city_clean'].apply(get_zone)
df['is_premium'] = (df['zone']==2).astype(int)
df['zone_score'] = (10-df['dist_mer'].clip(0,10)).round(2)
print('✅ Distances + zones créées')

✅ Distances + zones créées


In [12]:
# ══ 3. NLP FEATURES ══════════════════════════════════
NLP = {
    'has_standing':   ['standing','grand standing','luxe','haut standing','prestige'],
    'has_neuf':       ['neuf','nouvelle construction','tout neuf','livraison','cle en main'],
    'has_renove':     ['renov','refait','modernise','recent','retape'],
    'has_piscine':    ['piscine','pool','swimming'],
    'has_meuble':     ['meuble','equipe','amenage','all inclusive'],
    'has_clim':       ['clim','climatise','climatisation','air conditionne'],
    'has_ascenseur':  ['ascenseur','elevator'],
    'has_parking':    ['parking','garage','carport'],
    'has_jardin':     ['jardin','garden','verdure'],
    'has_balcon':     ['balcon','terrasse','loggia'],
    'has_gardien':    ['gardien','vigile','securite','gardiennage'],
    'has_groupe':     ['groupe electrogene','generateur','groupe elec'],
    'has_vue_mer':    ['vue mer','face mer','bord de mer'],
    'has_tf':         ['titre foncier','tf ','foncier regularise'],
    'has_viabilise':  ['viabilise','eau electricite','vrd'],
    'has_cuisine_am': ['cuisine americaine','cuisine integree','americaine'],
    'has_invest':     ['investissement','rendement','rentable','locatif'],
}

def nlp_row(row):
    text = ' '.join([str(row.get(c) or '') for c in ['title','description','property_type']]).lower()
    for a,b in [('é','e'),('è','e'),('ê','e'),('à','a'),('â','a'),('î','i'),('ô','o')]:
        text=text.replace(a,b)
    return {k: int(any(w in text for w in ws)) for k,ws in NLP.items()}

nlp_df = df.apply(lambda r: pd.Series(nlp_row(r)), axis=1)
df = pd.concat([df, nlp_df], axis=1)
PREST = ['has_standing','has_piscine','has_vue_mer','has_gardien','has_clim','has_tf','has_ascenseur']
df['prestige'] = df[PREST].sum(axis=1)
print('✅ NLP features créées')
print({c: f"{df[c].mean()*100:.0f}%" for c in list(NLP.keys())[:8]})

✅ NLP features créées
{'has_standing': '6%', 'has_neuf': '6%', 'has_renove': '0%', 'has_piscine': '5%', 'has_meuble': '30%', 'has_clim': '7%', 'has_ascenseur': '7%', 'has_parking': '9%'}


In [13]:
# ══ 4. FEATURES NUMÉRIQUES + IMPUTATION ═════════════
# Fusionner source + NLP
df['surface'] = df['surface_nlp'].combine_first(df['surface_area'])
df['bedrooms']= df['bedrooms_nlp'].combine_first(df['bedrooms'])
# Valider
df.loc[df['surface'] > 50_000,'surface'] = np.nan
df.loc[df['surface'] <= 0,'surface']     = np.nan
df.loc[df['bedrooms'] > 25,'bedrooms']   = np.nan
# Imputation par médiane groupe (type + zone)
for col, gcols in [('surface',['type_norm','zone']),('bedrooms',['type_norm'])]:
    med = df.groupby(gcols)[col].transform('median')
    glob= df[col].median()
    df[col] = df[col].fillna(med).fillna(glob)
# Bathrooms
df['bathrooms_f'] = pd.to_numeric(df.get('bathrooms'), errors='coerce')
df['bathrooms_f'] = df['bathrooms_f'].fillna((df['bedrooms']*0.5).clip(1).round())
# Flags imputés (utiles pour le modèle)
df['surf_imp'] = df['surface_area'].isna().astype(int)
df['bed_imp']  = (df.get('bedrooms')<1 if 'bedrooms' in df.columns else pd.Series([1]*len(df))).astype(int)
# Features dérivées
df['log_surf']   = np.log1p(df['surface'])
df['rooms']      = df['bedrooms'] + df['bathrooms_f']
df['surf_room']  = df['surface'] / df['rooms'].clip(1)
df['bath_bed']   = df['bathrooms_f'] / df['bedrooms'].clip(1)
df['surf_sq']    = df['surface'] ** 0.5
df['zone_surf']  = df['zone'] * df['surface']
# Interaction transaction × type (prix très différent selon vente/location)
df['is_location'] = (df['transaction']=='Location').astype(int)
print('✅ Features numériques prêtes')
print(f'   surface: {df.surface.notna().mean()*100:.0f}% | bedrooms: {df.bedrooms.notna().mean()*100:.0f}%')

✅ Features numériques prêtes
   surface: 100% | bedrooms: 100%


In [14]:
# ══ 5. TARGET ENCODING BAYÉSIEN ═════════════════════
df['log_price'] = np.log1p(df['price'])

def bayes_encode(df_sub, col, target='log_price', k=10):
    gm = df_sub[target].mean()
    st = df_sub.groupby(col)[target].agg(['mean','count'])
    st['enc'] = (st['count']*st['mean'] + k*gm) / (st['count']+k)
    return df_sub[col].map(st['enc']).fillna(gm)

# Encoder séparément Vente/Location
for txn in ['Vente','Location']:
    mask = df['transaction']==txn
    if mask.sum() < 50: continue
    s = df[mask].copy()
    df.loc[mask,'city_enc'] = bayes_encode(s,'city_clean').values
    df.loc[mask,'type_enc'] = bayes_encode(s,'type_norm').values

df['city_enc'] = df['city_enc'].fillna(df['log_price'].mean())
df['type_enc'] = df['type_enc'].fillna(df['log_price'].mean())
print('✅ Target encoding bayésien créé')

✅ Target encoding bayésien créé


In [15]:
# ══ 6. LISTE FEATURES + SAUVEGARDE ══════════════════
NUM = [
    'surface','log_surf','surf_sq','bedrooms','bathrooms_f',
    'rooms','surf_room','bath_bed','zone_surf',
    'lat','lon','zone','is_premium','zone_score',
    'dist_mer','dist_centre','dist_aeroport','dist_aibd','dist_port','dist_ucad','dist_vdn',
    'log_dist_mer','log_dist_centre',
    'city_enc','type_enc','is_location',
    'surf_imp','bed_imp','prestige',
] + list(NLP.keys())

CAT = ['type_norm','source']
FEATS = [f for f in NUM+CAT if f in df.columns]
missing = [f for f in NUM+CAT if f not in df.columns]
print(f'Features: {len(FEATS)}/{len(NUM+CAT)} disponibles')
if missing: print(f'Manquantes: {missing}')

os.makedirs('../data', exist_ok=True)
os.makedirs('../properties/ml', exist_ok=True)
df.to_csv('../data/dataset_features.csv', index=False)
cfg = {'NUM_FEATURES':[f for f in NUM if f in df.columns],
       'CAT_FEATURES':[f for f in CAT if f in df.columns],
       'TARGET':'log_price','n_features':len(FEATS),'n_samples':len(df)}
with open('../properties/ml/features_config.json','w') as f:
    import json; json.dump(cfg, f, indent=2)
print(f'✅ {len(df):,} annonces · {len(FEATS)} features sauvegardées')

Features: 48/48 disponibles
✅ 9,376 annonces · 48 features sauvegardées
